# Room EQ - Prototype

Below, we illustrate a prototype for our software version of a room-eq device. It does the following:

1. Generates Sine Sweep
2. Listens to Room IR from Sine Sweep
3. Computes Filter Coefficients from Estimated Room IR

In [ ]:
# Step 1: Import necessary libraries
import numpy as np


In [ ]:
# Step 2: Generate Sweep
#
# A "sine sweep" is a sine wave whose frequency rises over time.
# We use a *logarithmic* sweep (from Farina 2000) because human hearing
# is logarithmic — an octave from 100→200 Hz feels the same "distance"
# as an octave from 1000→2000 Hz, so we want to spend equal time in each.
#
# The sweep goes from frequency f1 (Hz) to f2 (Hz) over T seconds.
# At every moment t, the instantaneous frequency is:
#
#   f(t) = f1 * (f2/f1)^(t/T)
#
# A sine wave needs a *phase* argument, not an instantaneous frequency.
# Phase is the running total of all the angles we've passed through:
#
#   phase(t) = ∫₀ᵗ 2π·f(τ) dτ
#            = 2π · K · (e^(t/L) − 1)
#
# where K and L are just convenient shorthands derived by Farina (see below).

def generate_sweep(f1=20, f2=20000, T=5.0, fs=44100):
    """
    Generate a logarithmic sine sweep.

    Parameters
    ----------
    f1 : float  — start frequency in Hz  (default 20 Hz)
    f2 : float  — end   frequency in Hz  (default 20 000 Hz)
    T  : float  — duration in seconds    (default 5 s)
    fs : int    — sample rate in Hz      (default 44 100 Hz) - CD Quality 

    Returns
    -------
    t      : 1-D array of time values (seconds)
    sweep  : 1-D array of audio samples in [-1, 1]
    """

    # --- angular frequencies (radians per second) ---
    # Multiplying by 2π converts "cycles per second" (Hz) to
    # "radians per second", which is what the math below needs.
    w1 = 2 * np.pi * f1
    w2 = 2 * np.pi * f2

    # --- time axis ---
    # Create one sample for every 1/fs seconds, from t=0 up to (but not
    # including) t=T. e.g. fs=44100 → 44 100 samples per second.
    t = np.arange(0, T, 1.0 / fs)   # shape: (T*fs,)

    # --- Farina's helper constants ---
    # These come straight from evaluating the integral ∫ w1·(w2/w1)^(τ/T) dτ.
    # K has units of radians;  L has units of seconds.
    L = T / np.log(w2 / w1)         # "time scale" of the exponential growth
    K = T * w1 / np.log(w2 / w1)    # = w1 * L  (total phase scale factor)

    # --- instantaneous phase at every time sample ---
    # This is the closed-form result of the integral above.
    # Think of phase as the "angle" of the sine wave at each moment.
    # When phase grows slowly → low frequency.
    # When phase grows quickly → high frequency.
    phase = K * (np.exp(t / L) - 1.0)

    # --- the sweep signal ---
    # A pure sine wave is just sin(phase).  The phase here increases
    # exponentially, so the frequency starts low and rises to f2 at t=T.
    sweep = np.sin(phase)

    return t, sweep
